# Historical Binance 15-minute direction dataset

This is a separate learning slice. It does not rerun the completed recorder/archive pipeline.

\`\`\`text
historical Binance 1m klines
          ↓
fixed UTC quarter-hour decisions
          ↓
12 leakage-safe short-term features + next non-overlapping 15m label
          ↓
audit CSV + model-ready CSV + chronological train/validation/holdout report
\`\`\`

The public historical source has interval timestamps, not the original client receipt time. The dataset therefore records the \`interval_complete_assumption\`; it is not receipt-time verified. No GPU, Polymarket data, Chainlink data, trading logic, or model training is used in this notebook.


## Operating rules

- Colab is stateless. Git stores code; Drive stores raw CSVs, checkpoints, outputs, and reports.
- The downloader checkpoints one UTC day at a time and skips only verified day files.
- The dataset builder checkpoints one target day at a time and skips only verified 96-row day files.
- Missing observations remain represented by invalid audit rows; no prices are invented.
- Every feature uses only completed 1-minute bars before the decision time.
- A decision at \`10:15\` uses history through the \`10:14\` close and predicts \`10:15\` through \`10:29\`.
- The target price and label are audit information, never model feature columns.
- The proposed first chronological split is 20 training days, 4 validation days, and 5 final holdout days. Review that choice before running the builder cell.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from collections import Counter

REPOSITORY = 'https://github.com/matahariramadhan/tradingbot-data.git'
REVISION = '91507cf3303bc0a88977091c3601175b3acd21e4'
PROJECT_DIR = Path('/content/tradingbot_v2')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    assert (PROJECT_DIR / '.git').is_dir(), f'not a Git checkout: {PROJECT_DIR}'

subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '.[training]'], cwd=PROJECT_DIR, check=True)
sys.path.insert(0, str(PROJECT_DIR))

print('repository:', REPOSITORY)
print('revision:', subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True
).strip())


In [ ]:
from importlib.metadata import version as distribution_version
import tradingbot_data

assert distribution_version('tradingbot-data') == '0.9.0'
assert tradingbot_data.__version__ == '0.9.0'
help_text = subprocess.check_output(['tradingbot-data', '--help'], text=True)
assert 'historical-download' in help_text
assert 'historical-15m' in help_text
print('distribution version:', distribution_version('tradingbot-data'))
print('package version:', tradingbot_data.__version__)
print(help_text)


## 1. Mount Drive and define the durable artifact contract

The raw range includes one warm-up day, \`2026-06-29\`, for the first target day. The target period itself is \`2026-06-30\` through \`2026-07-28\` inclusive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONTROL_DIR = Path('/content/drive/MyDrive/tradingbot-data-audit')
HIST_RAW_DIR = CONTROL_DIR / 'historical-binance-1m-v1'
HIST_DATASET_DIR = CONTROL_DIR / 'historical-binance-15m-v1'
DOWNLOAD_CHECKPOINT = CONTROL_DIR / 'historical-binance-1m-download-v1.json'
DOWNLOAD_REPORT = CONTROL_DIR / 'historical-binance-1m-download-report-v1.json'
DATASET_CHECKPOINT = CONTROL_DIR / 'historical-binance-15m-build-v1.json'
DATASET_REPORT = CONTROL_DIR / 'historical-binance-15m-report-v1.json'
SPLIT_REPORT = CONTROL_DIR / 'historical-binance-15m-split-v1.json'
AUDIT_OUTPUT = HIST_DATASET_DIR / 'dataset-audit-v1.csv'
MODEL_OUTPUT = HIST_DATASET_DIR / 'model-ready-v1.csv'

RAW_START = '2026-06-29'
RAW_END = '2026-07-29'
TARGET_START = '2026-06-30'
TARGET_END = '2026-07-29'
TRAIN_DAY_COUNT = 20
VALIDATION_DAY_COUNT = 4

print('raw output:', HIST_RAW_DIR)
print('dataset output:', HIST_DATASET_DIR)
print('proposed split: 20 train days / 4 validation days / 5 holdout days')


## 2. Download the independent historical source

This cell may take time. If Colab stops, rerun it: the checkpoint and verified daily CSVs are on Drive, so completed days are skipped.


In [ ]:
subprocess.run([
    'tradingbot-data', 'historical-download',
    '--symbol', 'BTCUSDT',
    '--start-date', RAW_START,
    '--end-date', RAW_END,
    '--output-dir', str(HIST_RAW_DIR),
    '--checkpoint', str(DOWNLOAD_CHECKPOINT),
    '--report', str(DOWNLOAD_REPORT),
    '--request-delay-seconds', '0.1',
], cwd=PROJECT_DIR, check=True)


In [ ]:
assert DOWNLOAD_REPORT.is_file(), f'missing download report: {DOWNLOAD_REPORT}'
download_report = json.loads(DOWNLOAD_REPORT.read_text(encoding='utf-8'))
assert download_report['status'] == 'completed'
assert download_report['config']['interval'] == '1m'
assert download_report['config']['start_date'] == RAW_START
assert download_report['config']['end_date_exclusive'] == RAW_END
assert download_report['receipt_time_available'] is False
assert download_report['availability_policy'] == 'interval_complete_assumption'
print(json.dumps(download_report['totals'], indent=2))
print('downloaded day files:', len(download_report['days']))


## 3. Build the audited 15-minute dataset

The builder emits 96 fixed quarter-hour rows per target day. It retains target prices in the audit view for inspection but excludes them from the model-ready view.


In [ ]:
subprocess.run([
    'tradingbot-data', 'historical-15m',
    '--raw-dir', str(HIST_RAW_DIR),
    '--download-report', str(DOWNLOAD_REPORT),
    '--target-start', TARGET_START,
    '--target-end', TARGET_END,
    '--output-dir', str(HIST_DATASET_DIR),
    '--checkpoint', str(DATASET_CHECKPOINT),
    '--report', str(DATASET_REPORT),
    '--split-report', str(SPLIT_REPORT),
    '--train-day-count', str(TRAIN_DAY_COUNT),
    '--validation-day-count', str(VALIDATION_DAY_COUNT),
], cwd=PROJECT_DIR, check=True)


In [ ]:
assert DATASET_REPORT.is_file(), f'missing dataset report: {DATASET_REPORT}'
assert SPLIT_REPORT.is_file(), f'missing split report: {SPLIT_REPORT}'
dataset_report = json.loads(DATASET_REPORT.read_text(encoding='utf-8'))
split_report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
assert dataset_report['status'] == 'completed'
assert split_report['status'] == 'completed'
assert split_report['verification']['train_validation_holdout_overlap_keys'] == 0
assert split_report['verification']['model_keys_unique'] is True
assert split_report['verification']['chronological_model_keys'] is True
print('dataset report:', DATASET_REPORT)
print(json.dumps(dataset_report['totals'], indent=2))
print('split report:', SPLIT_REPORT)
print(json.dumps(split_report['totals'], indent=2))


In [ ]:
with MODEL_OUTPUT.open(newline='', encoding='utf-8') as source:
    model_reader = csv.DictReader(source)
    model_rows = list(model_reader)
with AUDIT_OUTPUT.open(newline='', encoding='utf-8') as source:
    audit_reader = csv.DictReader(source)
    audit_columns = audit_reader.fieldnames

assert model_reader.fieldnames is not None
assert 'label' in model_reader.fieldnames
assert 'target_start_price' not in model_reader.fieldnames
assert 'target_end_price' not in model_reader.fieldnames
assert 'target_return_15m' not in model_reader.fieldnames
assert audit_columns is not None and 'target_end_price' in audit_columns
assert len(model_rows) == dataset_report['totals']['model_rows_usable']
print('audit columns:', audit_columns)
print('model columns:', model_reader.fieldnames)
print('model-ready rows:', len(model_rows))


## 4. Human-readable dataset review

These plots reload the durable model-ready CSV. They do not train a model and do not use future target prices as features.


In [ ]:
import matplotlib.pyplot as plt

feature_columns = [
    'return_1m', 'return_5m', 'return_15m', 'return_30m',
    'volatility_5m', 'volatility_15m', 'volume_ratio_5m',
    'candle_body_5m', 'high_low_range_5m', 'distance_ma_15',
    'ma_slope_15', 'rsi_14',
]
label_counts = Counter(row['label'] for row in model_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(sorted(label_counts), [label_counts[label] for label in sorted(label_counts)])
axes[0].set_title('15-minute label counts')
axes[0].set_ylabel('rows')
for field in feature_columns:
    values = [float(row[field]) for row in model_rows]
    axes[1].plot(sorted(values), label=field, alpha=0.8)
axes[1].set_title('Sorted feature values')
axes[1].set_xlabel('sorted row index')
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## What this gate means

At this point we have not shown predictive skill. We have shown a reproducible, auditable supervised-learning table with an explicit availability assumption, no feature/target overlap, and a chronological split.

The next lesson is to interpret the usable-row counts, inspect the invalid-row reasons, and decide whether the proposed 20/4/5 split is appropriate before training a simple baseline.
